In [ ]:
# Colab setup. In a local clone you can comment this out.
!pip install -q git+https://github.com/2forts/qcirclab_repo.git

In [ ]:
import numpy as np
import time
from collections import deque

from qcirclab import Circuit
import qcirclab.gates as qg
from qcirclab.operators import (
    append_operation,
    circuit_unitary,
    equal_up_to_global_phase,
    circuit_without_measurements,
)
from qcirclab.metrics import circuit_metrics, print_metrics

## Subsection 7.2.5 **Practical metric evaluation**

In [ ]:
# Example circuit: three qubits and a final measurement layer.

qc = Circuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.barrier()
qc.rx(0.4, 0)
qc.ry(0.2, 1)
qc.rz(0.1, 2)
qc.measure_all()

print(qc.draw())
print_metrics("Logical circuit", qc)

In [ ]:
# A simple backend-independent timing estimate.
# These numbers are illustrative only.

gate_durations_ns = {
    "h": 35,
    "x": 35,
    "rx": 35,
    "ry": 35,
    "rz": 0,
    "s": 0,
    "t": 0,
    "cx": 300,
    "cz": 300,
    "swap": 900,
    "measure": 1000,
}

def estimate_duration_ns(qc, durations):
    qtime = [0.0] * qc.n_qubits
    ctime = [0.0] * qc.n_clbits

    for op in qc.operations:
        if op.name == "barrier":
            m = max(qtime + ctime) if (qtime or ctime) else 0
            qtime = [m] * qc.n_qubits
            ctime = [m] * qc.n_clbits
            continue

        used_q = set(op.targets) | set(op.controls)
        used_c = set(op.ctargets)
        if op.condition is not None:
            used_c.add(op.condition.bit)

        start = 0.0
        if used_q:
            start = max(start, max(qtime[q] for q in used_q))
        if used_c:
            start = max(start, max(ctime[c] for c in used_c))

        duration = durations.get(op.name, 50)
        finish = start + duration

        for q in used_q:
            qtime[q] = finish
        for c in used_c:
            ctime[c] = finish

    return max(qtime + ctime) if (qtime or ctime) else 0.0

print("Estimated logical duration (ns):", estimate_duration_ns(qc, gate_durations_ns))

## Subsection 7.3.1 **Logical vs. routed circuits**

In [ ]:
# Logical circuit with a long-range interaction.

logical = Circuit(3, 3)
logical.h(0)
logical.cx(0, 2)
logical.cx(1, 2)
logical.barrier()
logical.measure_all()

print("Logical circuit:")
print(logical.draw())
print_metrics("Logical", logical)

In [ ]:
def shortest_path(num_qubits, edges, start, goal):
    adj = {q: [] for q in range(num_qubits)}
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)

    queue = deque([(start, [start])])
    seen = {start}

    while queue:
        node, path = queue.popleft()
        if node == goal:
            return path
        for nxt in adj[node]:
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))

    raise ValueError(f"No path between qubits {start} and {goal}")


def has_edge(edges, a, b):
    return (a, b) in edges or (b, a) in edges


def route_cx_to_coupling(qc: Circuit, edges) -> Circuit:
    # Toy router: replace non-adjacent CX gates by SWAP chains on an undirected topology.
    routed = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_routed")

    for op in qc.operations:
        if op.name == "cx" and len(op.controls) == 1 and len(op.targets) == 1:
            c = op.controls[0]
            t = op.targets[0]

            if has_edge(edges, c, t):
                routed.cx(c, t)
            else:
                path = shortest_path(qc.n_qubits, edges, c, t)

                # Move the control state next to the target, apply CX, then restore.
                for i in range(len(path) - 2):
                    routed.swap(path[i], path[i + 1])

                routed.cx(path[-2], path[-1])

                for i in reversed(range(len(path) - 2)):
                    routed.swap(path[i], path[i + 1])
        else:
            append_operation(routed, op)

    return routed


line_edges_3 = [(0, 1), (1, 2)]

physical = route_cx_to_coupling(logical, line_edges_3)

print("Physical/routed circuit on a line topology:")
print(physical.draw())

print_metrics("Logical", logical)
print_metrics("Routed", physical)

## Subsection 7.3.2 **A simple pass-based optimization pipeline**

In [ ]:
# Circuit with redundant and simplifiable operations.

qc_pass = Circuit(2, 2)
qc_pass.h(0)
qc_pass.h(0)          # cancels
qc_pass.rx(0.3, 0)
qc_pass.rx(-0.3, 0)   # cancels
qc_pass.rz(0.2, 0)
qc_pass.rz(0.4, 0)    # fuses with previous Rz
qc_pass.cx(0, 1)
qc_pass.cx(0, 1)      # cancels
qc_pass.barrier()
qc_pass.measure_all()

print("Original:")
print(qc_pass.draw())
print_metrics("Original", qc_pass)

In [ ]:
SELF_INVERSE = {"h", "x", "y", "z", "cx", "cz", "swap"}

def same_location(op1, op2):
    return (
        op1.name == op2.name
        and op1.targets == op2.targets
        and op1.controls == op2.controls
        and op1.condition == op2.condition
    )


def cancel_adjacent_gates(qc: Circuit) -> Circuit:
    stack = []

    for op in qc.operations:
        if op.name == "barrier":
            continue

        if stack:
            prev = stack[-1]

            if (
                op.name in SELF_INVERSE
                and prev.name in SELF_INVERSE
                and same_location(prev, op)
            ):
                stack.pop()
                continue

            if (
                op.name in {"rx", "ry", "rz"}
                and prev.name == op.name
                and prev.targets == op.targets
                and prev.controls == op.controls
                and prev.params
                and op.params
                and abs(prev.params[0] + op.params[0]) < 1e-12
            ):
                stack.pop()
                continue

        stack.append(op)

    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cancelled")
    for op in stack:
        append_operation(out, op)
    return out


def fuse_adjacent_rotations(qc: Circuit) -> Circuit:
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_fused")
    i = 0
    ops = qc.operations

    while i < len(ops):
        op = ops[i]

        if (
            op.name in {"rx", "ry", "rz"}
            and len(op.targets) == 1
            and not op.controls
            and op.params
        ):
            axis = op.name
            q = op.targets[0]
            theta = op.params[0]
            j = i + 1

            while j < len(ops):
                nxt = ops[j]
                if (
                    nxt.name == axis
                    and nxt.targets == (q,)
                    and not nxt.controls
                    and nxt.params
                ):
                    theta += nxt.params[0]
                    j += 1
                else:
                    break

            if abs(theta) > 1e-12:
                getattr(out, axis)(theta, q)

            i = j
            continue

        append_operation(out, op)
        i += 1

    return out


def toy_pass_pipeline(qc: Circuit, level=1) -> Circuit:
    out = qc.copy()

    if level >= 1:
        out = cancel_adjacent_gates(out)

    if level >= 2:
        out = fuse_adjacent_rotations(out)

    return out


for level in range(3):
    opt = toy_pass_pipeline(qc_pass, level=level)
    print_metrics(f"Pipeline level {level}", opt)
    print(opt.draw())

## Subsection **7.4.1 Gate cancellation and commutation rules**

In [ ]:
# Redundant X gates separated by operations on another qubit.

qc_comm = Circuit(2, 2)

qc_comm.x(0)
qc_comm.h(1)
qc_comm.rz(0.3, 1)
qc_comm.x(0)

qc_comm.barrier()
qc_comm.measure_all()

print("Before optimization:")
print(qc_comm.draw())
print_metrics("Before", qc_comm)

# In this small example, the gates on qubit 1 commute with
# the X gates on qubit 0. A commutation-aware optimizer may
# reorder them and then cancel the two X gates.

## Subsection **7.4.2 Gate fusion and parameter merging**

In [ ]:
# Consecutive rotations around the same axis can be fused.

qc_fuse = Circuit(2, 2)

qc_fuse.rz(0.2, 0)
qc_fuse.rz(0.4, 0)
qc_fuse.rx(0.1, 1)
qc_fuse.rx(0.3, 1)
qc_fuse.ry(0.5, 0)

qc_fuse.barrier()
qc_fuse.measure_all()

print("Before fusion:")
print(qc_fuse.draw())
print_metrics("Before", qc_fuse)

qc_fused = Circuit(2, 2)

qc_fused.rz(0.6, 0)
qc_fused.rx(0.4, 1)
qc_fused.ry(0.5, 0)

qc_fused.barrier()
qc_fused.measure_all()

print("After fusion:")
print(qc_fused.draw())
print_metrics("After", qc_fused)

## Subsection **7.4.4 Native gate sets and basis decomposition**

In [ ]:
def decompose_to_rx_rz_cx(qc: Circuit) -> Circuit:
    # Rewrite selected logical gates into the basis
    # {rx, rz, cx, measure}.

    out = Circuit(
        qc.n_qubits,
        qc.n_clbits,
        name=qc.name + "_basis"
    )

    for op in qc.operations:

        if op.name in {"barrier", "measure", "reset"}:
            append_operation(out, op)

        elif op.name == "h":
            q = op.targets[0]

            # H up to global phase.
            out.rz(np.pi/2, q)
            out.rx(np.pi/2, q)
            out.rz(np.pi/2, q)

        elif op.name == "x":
            out.rx(np.pi, op.targets[0])

        elif op.name == "z":
            out.rz(np.pi, op.targets[0])

        elif op.name == "s":
            out.rz(np.pi/2, op.targets[0])

        elif op.name == "t":
            out.rz(np.pi/4, op.targets[0])

        elif op.name == "ry":
            q = op.targets[0]
            theta = op.params[0]

            # RY(theta) = RZ(pi/2) RX(theta) RZ(-pi/2)
            # up to a global phase.
            out.rz(np.pi/2, q)
            out.rx(theta, q)
            out.rz(-np.pi/2, q)

        elif op.name == "cz":
            c = op.controls[0]
            t = op.targets[0]

            out.rz(np.pi/2, t)
            out.rx(np.pi/2, t)
            out.rz(np.pi/2, t)

            out.cx(c, t)

            out.rz(np.pi/2, t)
            out.rx(np.pi/2, t)
            out.rz(np.pi/2, t)

        elif op.name in {"rx", "rz", "cx"}:
            append_operation(out, op)

        else:
            append_operation(out, op)

    return out

In [ ]:
logical_basis = Circuit(2, 2)

logical_basis.h(0)
logical_basis.ry(0.5, 1)
logical_basis.cz(0, 1)

logical_basis.measure_all()

basis_circuit = decompose_to_rx_rz_cx(logical_basis)

print("Logical circuit:")
print(logical_basis.draw())
print_metrics("Logical", logical_basis)

print("Basis-decomposed circuit:")
print(basis_circuit.draw())
print_metrics("Basis-decomposed", basis_circuit)

## Subsection **7.5.1 Topology-aware layout selection**

In [ ]:
# Compare candidate layouts using weighted interaction distance.

def interaction_weights(qc: Circuit):
    weights = {}

    for op in qc.operations:
        if (
            op.name == "cx"
            and len(op.controls) == 1
            and len(op.targets) == 1
        ):
            a = op.controls[0]
            b = op.targets[0]
            key = tuple(sorted((a, b)))
            weights[key] = weights.get(key, 0) + 1

    return weights


def layout_cost(weights, layout, edges, num_physical):
    cost = 0

    for (a, b), weight in weights.items():
        pa = layout[a]
        pb = layout[b]

        path = shortest_path(
            num_physical,
            edges,
            pa,
            pb
        )

        cost += weight * (len(path) - 1)

    return cost


qc_layout = Circuit(5, 5)

qc_layout.h(0)
qc_layout.cx(0, 4)
qc_layout.cx(0, 4)
qc_layout.cx(1, 2)

qc_layout.barrier()
qc_layout.measure_all()

line_edges_5 = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),
]

weights = interaction_weights(qc_layout)

layout_bad = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

layout_good = {
    0: 1,
    1: 3,
    2: 4,
    3: 0,
    4: 2,
}

print("Interaction weights:")
print(weights)

print(
    "Bad layout cost:",
    layout_cost(
        weights,
        layout_bad,
        line_edges_5,
        5
    )
)

print(
    "Good layout cost:",
    layout_cost(
        weights,
        layout_good,
        line_edges_5,
        5
    )
)

## Subsection **7.5.2 Routing trade-offs and SWAP insertion**

In [ ]:
# Routing trade-off on a four-qubit line:
# 0 -- 1 -- 2 -- 3
#
# Initial logical-to-physical layout:
# q0 -> 0
# q1 -> 3
# q2 -> 2
# q3 -> 1
#
# Logical interactions to execute:
# CX(q0, q1), then CX(q1, q2)

# Strategy 1: restore the layout after routing CX(q0, q1).

restore_route = Circuit(4)

# Move q0 from physical 0 to physical 2,
# next to q1 on physical 3.
restore_route.swap(0, 1)
restore_route.swap(1, 2)

# Execute CX(q0, q1) as CX(2, 3).
restore_route.cx(2, 3)

# Restore the original layout.
restore_route.swap(1, 2)
restore_route.swap(0, 1)

# Now q1 is again on physical 3 and q2 on physical 2,
# so CX(q1, q2) is local.
restore_route.cx(3, 2)

print("Routing with layout restoration:")
print(restore_route.draw())
print_metrics("Restore layout", restore_route)


# Strategy 2: keep the updated layout after routing CX(q0, q1).

keep_route = Circuit(4)

# Move q0 from physical 0 to physical 2,
# next to q1 on physical 3.
keep_route.swap(0, 1)
keep_route.swap(1, 2)

# Execute CX(q0, q1) as CX(2, 3).
keep_route.cx(2, 3)

# Do not undo the SWAPs.
# The layout has changed. Now q2 is on physical 1.
# To execute CX(q1, q2), move q2 from physical 1 to 2.
keep_route.swap(1, 2)

# Execute CX(q1, q2) as CX(3, 2).
keep_route.cx(3, 2)

print("Routing with updated layout:")
print(keep_route.draw())
print_metrics("Keep updated layout", keep_route)

## Subsection **7.5.3 Calibration-aware layout and gate selection**

In [ ]:
# Fake calibration data for a five-qubit line.
# Values are illustrative only.

fake_calibration = {
    "single_qubit_error": {
        0: 0.0008,
        1: 0.0011,
        2: 0.0005,
        3: 0.0014,
        4: 0.0007,
    },
    "readout_error": {
        0: 0.020,
        1: 0.025,
        2: 0.015,
        3: 0.030,
        4: 0.018,
    },
    "cx_error": {
        (0, 1): 0.012,
        (1, 2): 0.045,  # noisy coupling
        (2, 3): 0.009,
        (3, 4): 0.015,
    },
}


def edge_error(calibration, a, b):
    key = (a, b)

    if key not in calibration["cx_error"]:
        key = (b, a)

    return calibration["cx_error"][key]


def path_cx_error(calibration, path):
    total = 0.0

    for a, b in zip(path[:-1], path[1:]):
        total += edge_error(calibration, a, b)

    return total


path_short = [0, 1, 2]
path_long = [0, 1, 2, 3, 4]

print("Short path:", path_short)
print(
    "Estimated CX error:",
    path_cx_error(fake_calibration, path_short)
)

print("Longer path:", path_long)
print(
    "Estimated CX error:",
    path_cx_error(fake_calibration, path_long)
)

## Subsection **7.6.1 Checking functional equivalence**

In [ ]:
# Functional equivalence check for a local rewrite.

qc_original = Circuit(1)

qc_original.h(0)
qc_original.x(0)
qc_original.h(0)

qc_optimized = Circuit(1)

qc_optimized.z(0)

print("Original circuit:")
print(qc_original.draw())
print_metrics("Original", qc_original)

print("Optimized circuit:")
print(qc_optimized.draw())
print_metrics("Optimized", qc_optimized)

U_original = circuit_unitary(qc_original)
U_optimized = circuit_unitary(qc_optimized)

print(
    "Equivalent up to global phase:",
    equal_up_to_global_phase(
        U_original,
        U_optimized
    )
)

## Subsection **7.6.2 Before-and-after metric comparison**

In [ ]:
qc_compare = Circuit(2, 2)

qc_compare.h(0)
qc_compare.h(0)
qc_compare.rx(0.3, 0)
qc_compare.rx(-0.3, 0)
qc_compare.rz(0.2, 1)
qc_compare.rz(0.4, 1)
qc_compare.cx(0, 1)
qc_compare.cx(0, 1)

qc_compare.barrier()
qc_compare.measure_all()

qc_compare_opt = toy_pass_pipeline(
    qc_compare,
    level=2
)

print("Original circuit:")
print(qc_compare.draw())
print_metrics("Original", qc_compare)

print("Optimized circuit:")
print(qc_compare_opt.draw())
print_metrics("Optimized", qc_compare_opt)

##Subsection **7.6.3 Noise-aware validation**

In [ ]:
def estimated_fidelity_from_logical_ops(
    qc,
    oneq_error=0.001,
    twoq_error=0.015,
    multiq_error=0.05,
):
    fidelity = 1.0

    for op in qc.operations:

        if op.name in {"barrier", "measure", "reset"}:
            continue

        arity = len(op.targets) + len(op.controls)

        if arity == 1:
            fidelity *= (1 - oneq_error)

        elif arity == 2:
            fidelity *= (1 - twoq_error)

        else:
            fidelity *= (1 - multiq_error)

    return fidelity

In [ ]:
qc_noisy_orig = Circuit(2, 2)

qc_noisy_orig.h(0)
qc_noisy_orig.cx(0, 1)
qc_noisy_orig.cx(0, 1)
qc_noisy_orig.rz(0.4, 0)

qc_noisy_orig.barrier()
qc_noisy_orig.measure_all()

qc_noisy_opt = Circuit(2, 2)

qc_noisy_opt.h(0)
qc_noisy_opt.rz(0.4, 0)

qc_noisy_opt.barrier()
qc_noisy_opt.measure_all()

print("Original circuit:")
print(qc_noisy_orig.draw())
print_metrics("Original", qc_noisy_orig)

print("Optimized circuit:")
print(qc_noisy_opt.draw())
print_metrics("Optimized", qc_noisy_opt)

print(
    "Estimated original fidelity:",
    estimated_fidelity_from_logical_ops(qc_noisy_orig)
)

print(
    "Estimated optimized fidelity:",
    estimated_fidelity_from_logical_ops(qc_noisy_opt)
)